# データベース演習 第9回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- **「編集者」ではなく「閲覧者」**に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

### 使用するデータベースのダウンロード

* 次のセルで私のgithubリポジトリにあるSQLite3のデータベースをダウンロードしています
  * syllabusデータベース
  * weblogデータベース（テーブル3個のみ）
* ダウンロードは，（私の大学の環境では）10秒以内に完了します
* Colabのランタイムが切り替わると，その度にダウンロードする必要があります

In [ ]:
# SQLiteファイルをダウンロード

# syllabusデータベース
!curl -sLO https://raw.githubusercontent.com/ggszk/ggszk-lab-public/refs/heads/main/db/syllabus.sqlite3

# weblogデータベース（テーブル3個のみ）
!curl -sLO https://raw.githubusercontent.com/ggszk/ggszk-lab-public/refs/heads/main/db/weblog.sqlite3

### JupySQLのインストールと有効化

In [ ]:
# Colabではjupysqlのインストールが必要
import sys
if 'google.colab' in sys.modules:
    %pip install -q jupysql

In [ ]:
# jupysqlの拡張機能を有効化`
%load_ext sql

### シラバスデータベース

In [ ]:
# シラバスデータベースに接続する
%sql sqlite:///syllabus.sqlite3

### 教員テーブルの確認
* 初期状態ではデータは入っていない．

In [ ]:
%%sql
SELECT * FROM 教員;


## サンプルプログラム9-1：データの作成

In [ ]:
# サンプルプログラム9-1：データの作成

import sqlite3

connection = sqlite3.connect('syllabus.sqlite3')
cursor = connection.cursor()

sql = "INSERT INTO 教員 (氏名, 年齢) VALUES('鈴木源吾', 25)"
cursor.execute(sql)

connection.commit()

cursor.close()
connection.close()
print("プログラム終了")

### 次のセルは教員にデータができたかの確認用

In [ ]:
%%sql
SELECT * FROM 教員;

## サンプルプログラム9-2：コミットしなかったら何が起こるのか？
* 次のセルでデータの中身を確認

In [ ]:
# サンプルプログラム9-2：コミットしなかったら何が起こるのか？

import sqlite3

connection = sqlite3.connect('syllabus.sqlite3')
cursor = connection.cursor()

sql = "INSERT INTO 教員 (氏名, 年齢) VALUES('柄沢直之', 24)"
cursor.execute(sql)

#connection.commit()

cursor.close()
connection.close()
print("プログラム終了")

In [ ]:
%%sql
SELECT * FROM 教員;

## サンプルプログラム9-3：自動コミット（オートコミット）
* SQLiteでは，isolation_levelをNoneにすると自動コミット

In [ ]:
# サンプルプログラム9-3：自動コミット（オートコミット）

import sqlite3

connection = sqlite3.connect('syllabus.sqlite3', isolation_level=None)
cursor = connection.cursor()

sql = "INSERT INTO 教員 (氏名, 年齢) VALUES('柄沢直之', 24)"
cursor.execute(sql)

cursor.close()
connection.close()
print("プログラム終了")

### データの確認

In [ ]:
%%sql
SELECT * FROM 教員;

## サンプルプログラム9-4：INSERT文でのパラメータの利用

In [ ]:
# サンプルプログラム9-4：INSERT文でのパラメータの利用

import sqlite3

connection = sqlite3.connect('syllabus.sqlite3')
cursor = connection.cursor()

# 指定方法1
cursor.execute("INSERT INTO 教員 (氏名, 年齢) VALUES (?, ?);", ("𠮷田貴裕", 23))
# 指定方法2
#cursor.execute("INSERT INTO 教員 (氏名, 年齢) VALUES (:name, :age);", {"name":"𠮷田貴裕", "age":23})

connection.commit()

cursor.close()
connection.close()
print("プログラム終了")

### データの確認

In [ ]:
%%sql
SELECT * FROM 教員;

## 演習：データの作成（課題4　第2問）
* 配布資料の演習をやってみよう
* 次のセルの先頭で氏名と年齢を変数に代入し，syllabusデータベースの教員テーブルに行を1つ作成するプログラムを記述しよう
* パラメータ化（?記号 または :名前 形式）を使うこと（SQL文へ値を文字列連結しない）
* 変数の値を書き換えて再実行し，異なる氏名・年齢でも作成できることを確認しよう

In [ ]:
# 第2問：セルの先頭で氏名・年齢を変数に代入してから，パラメータ化（?記号）でINSERTする
# name = "田中一郎"
# age = 26
# 以下に続きを記述せよ


In [ ]:
%%sql
SELECT * FROM 教員;


## サンプルプログラム9-5：SQLAlchemyの基本
* with文を利用すると，コネクションのcloseは自動的に最後に呼び出される

In [ ]:
# サンプルプログラム9-5：SQLAlchemyの基本

from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///syllabus.sqlite3')

# コネクションオブジェクトを作って接続
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT * FROM 授業科目 WHERE 対象学年 = :year"), {"year":1})
    # resultはリストではない（ResultProxy），fetchoneが内部的に呼ばれている
    for row in result:
        print(row)


### 参考：with文を利用しない方法

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///syllabus.sqlite3')

connection = engine.connect()
result = connection.execute(
    text("SELECT * FROM 授業科目 WHERE 対象学年 = :year"), {"year":1})
for row in result:
    print(row)
connection.close()


## サンプルプログラム9-6：モデルの定義
* 次のセルを実行して，model.pyファイルを作成する
* モデルクラスの定義：クラスとテーブルの対応も定義される

In [ ]:
%%writefile model.py
#
# サンプルプログラム9-6：モデルクラスの定義（SQLAlchemy 2.0系）
#
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

# モデルベースクラスの作成（空のクラス）
class Base(DeclarativeBase):
    pass

# Teacherモデルクラス
class Teacher(Base):
    __tablename__ = "teacher"

    # 型ヒントとmapped_column()でカラムを定義
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column()
    age: Mapped[int] = mapped_column()

# 複数のクラスを作成したかったら，同様に定義していく


## サンプルプログラム9-7：ORマッピング．テーブルを作成

In [ ]:
# サンプルプログラム9-7：ORマッピング．テーブルを作成

from sqlalchemy import create_engine

from model import Base

# データベース（テーブル）作成
engine=create_engine('sqlite:///syllabus.sqlite3')
meta = Base.metadata
meta.create_all(engine)
print("プログラム終了")

### テーブルの確認
* テーブルteacherが作成されていることを確認する

In [ ]:
%%sql
SELECT * FROM sqlite_master;

## サンプルプログラム9-8：ORマッピング．データの追加


In [ ]:
# サンプルプログラム9-8：ORマッピング．データの追加

from model import Teacher
from sqlalchemy import create_engine
from sqlalchemy.orm import Session

# エンジン生成
engine = create_engine("sqlite:///syllabus.sqlite3")

# セッションをwith文で生成（終了時に自動でクローズ）
with Session(engine) as session:
    # データ追加 3人のデータを追加する
    t_obj_1 = Teacher(name="Suzuki", age=25)
    session.add(t_obj_1)
    t_obj_2 = Teacher(name="Karasawa", age=24)
    session.add(t_obj_2)
    t_obj_3 = Teacher(name="Yoshida", age=23)
    session.add(t_obj_3)

    session.commit()

print("プログラム終了")


データの確認
* 3人分のデータが追加されていることを確認する

In [ ]:
%%sql
SELECT * FROM teacher;

## サンプルプログラム9-9：データ検索（全件検索）

In [ ]:
# サンプルプログラム9-9：データ検索（全件検索）

from model import Teacher
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session

# エンジン生成
engine = create_engine("sqlite:///syllabus.sqlite3")

with Session(engine) as session:
    # 検索：select文を作成し，scalars()でTeacherオブジェクトを取り出す
    stmt = select(Teacher)
    teachers = session.scalars(stmt).all()
    for teacher in teachers:
        print(teacher.name)


## サンプルプログラム9-10：データ検索（条件検索）

In [ ]:
# サンプルプログラム9-10：データ検索（条件検索）

from model import Teacher
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session

# エンジン生成
engine = create_engine("sqlite:///syllabus.sqlite3")

with Session(engine) as session:
    # 検索：where() で条件を指定（年齢が24以下）
    stmt = select(Teacher).where(Teacher.age <= 24)
    teachers = session.scalars(stmt).all()
    for teacher in teachers:
        print(teacher.name)


## 演習：ORマッピング機能を用いたデータの作成（課題4　第3問）
* 配布資料の演習をやってみよう
* 次のセルの先頭で氏名と年齢を変数に代入し，SQLAlchemyの**ORマッピング機能を用いて** teacherテーブルに行を作成するプログラムを記述しよう（SQL文を直接実行する方法は不可）
* 変数の値を書き換えて再実行し，異なる氏名・年齢でも作成できることを確認しよう

In [ ]:
# 第3問：セルの先頭で氏名・年齢を変数に代入してから，ORマッピング（Teacher(...)）でデータを作成する
# name = "田中一郎"
# age = 26
# 以下に続きを記述せよ


### 次のセルを実行して，作成されたデータが正しいか確認すること


In [ ]:
%%sql
SELECT * FROM teacher;

## 以下の3つのセルは，教員テーブル・teacherテーブルの初期化用．
* やり直しするときに適宜使用すること．順に，
  * 教員テーブルのデータを空にする
  * teacherテーブルのデータを空にする
  * teacherテーブルを削除する

In [ ]:
%%sql
DELETE FROM 教員;

In [ ]:
%%sql
DELETE FROM teacher;

In [ ]:
%%sql
DROP TABLE teacher;